# Website Traffic Seasonality Analysis

A comprehensive exploration of web traffic patterns, trends, campaign effects, traffic anomalies, and source mix over a two-year period to support digital marketing and infrastructure decisions.

## Project Overview

This notebook analyzes two years of synthetic daily website metrics for a media or e-commerce website. We explore traffic trends, day-of-week seasonality, traffic source contributions, simulated campaign spikes, and anomaly detection. The goal is to help digital marketers and web analysts understand traffic dynamics before building attribution or forecasting models.

## Learning Objectives

- Build a realistic multi-column website metrics dataset with trend and seasonality
- Identify day-of-week patterns in web traffic
- Detect and quantify marketing campaign effects
- Analyze traffic source composition over time
- Flag anomalies using statistical thresholds
- Visualize bounce rate and session duration trends

## Business or Research Problem

A digital marketing team needs to understand its website's traffic dynamics to:
- Allocate budget to the most effective channels
- Time campaigns to amplify existing traffic peaks
- Detect traffic anomalies (bot attacks, server errors, viral content)
- Forecast server capacity requirements

## Why This Analysis Matters

- **Budget allocation**: Knowing which source drives quality traffic (low bounce, high session duration) prevents wasted spend
- **Campaign measurement**: Isolating campaign lift from organic trends requires baseline understanding
- **Capacity planning**: Traffic spikes can crash underpowered servers; forecasting peaks prevents downtime
- **SEO health**: Organic traffic trends reveal whether content and SEO investments are paying off

## Dataset Overview

**730 rows** (2 years: 2022–2023), one row per day.

| Column | Description |
|---|---|
| `date` | Calendar date |
| `sessions` | Total website sessions |
| `pageviews` | Total pageviews |
| `bounce_rate` | Fraction of single-page visits (0–1) |
| `avg_session_duration` | Avg time on site (minutes) |
| `new_users` | New visitor sessions |
| `source` | Dominant traffic source that day |

Sessions are higher on Mondays and during three simulated campaign periods. A background growth trend simulates organic channel growth.

## Dataset Source and License Notes

Fully synthetic dataset generated within this notebook. Inspired by typical SaaS/media analytics patterns (Google Analytics style). No real data included. Free for educational use.

## Environment Setup

```bash
pip install pandas numpy matplotlib seaborn scipy
```

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10')
print('Libraries loaded.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

START_DATE     = '2022-01-01'
END_DATE       = '2023-12-31'
BASE_SESSIONS  = 8000
TREND_DAILY    = 2.5         # organic growth sessions/day
MONDAY_LIFT    = 0.25        # 25% more sessions on Mondays
CAMPAIGN_LIFT  = 0.60        # 60% lift during campaigns
NOISE_STD      = 600

# Three campaign windows (start date, duration days)
CAMPAIGNS = [
    ('2022-03-01', 14),
    ('2022-09-15', 21),
    ('2023-11-20', 28),  # Black Friday / holiday campaign
]

SOURCES = ['organic', 'paid', 'direct', 'referral']
SOURCE_PROBS = [0.45, 0.25, 0.20, 0.10]

print(f'Period: {START_DATE} to {END_DATE}')
print(f'Base sessions: {BASE_SESSIONS:,} | Trend: +{TREND_DAILY}/day')

## Dataset Download or Loading

Generate the synthetic daily traffic dataset with trend, day-of-week effects, campaign spikes, and realistic engagement metrics.

In [ ]:
dates = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
n = len(dates)
t = np.arange(n)

# Trend + Monday lift
trend = BASE_SESSIONS + TREND_DAILY * t
monday_effect = np.where(dates.dayofweek == 0, MONDAY_LIFT * BASE_SESSIONS, 0)

# Weekend dip
weekend_dip = np.where(dates.dayofweek >= 5, -0.30 * BASE_SESSIONS, 0)

# Campaign flags
campaign_flag = np.zeros(n)
for start_str, dur in CAMPAIGNS:
    s = pd.Timestamp(start_str)
    for i, d in enumerate(dates):
        if s <= d < s + pd.Timedelta(days=dur):
            campaign_flag[i] = 1

campaign_effect = campaign_flag * CAMPAIGN_LIFT * BASE_SESSIONS

# Noise
noise = np.random.normal(0, NOISE_STD, n)

sessions = trend + monday_effect + weekend_dip + campaign_effect + noise
sessions = np.maximum(sessions, 500).round(0).astype(int)

# Derived metrics
pageviews = (sessions * np.random.uniform(2.5, 4.5, n)).round(0).astype(int)
bounce_rate = np.clip(0.55 - 0.10 * campaign_flag + np.random.normal(0, 0.05, n), 0.2, 0.85)
avg_session_dur = np.clip(3.5 + 1.5 * campaign_flag + np.random.normal(0, 0.5, n), 1.0, 10.0)
new_users = (sessions * np.random.uniform(0.55, 0.70, n)).round(0).astype(int)
source = np.random.choice(SOURCES, n, p=SOURCE_PROBS)

df = pd.DataFrame({
    'date'                : dates,
    'sessions'            : sessions,
    'pageviews'           : pageviews,
    'bounce_rate'         : bounce_rate.round(3),
    'avg_session_duration': avg_session_dur.round(2),
    'new_users'           : new_users,
    'source'              : source,
    'campaign_flag'       : campaign_flag.astype(int)
})
df.set_index('date', inplace=True)

print(f'Shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print('=== Validation ===')
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Sessions range : {df["sessions"].min():,} – {df["sessions"].max():,}')
print(f'Bounce range   : {df["bounce_rate"].min():.2f} – {df["bounce_rate"].max():.2f}')
print(f'Campaign days  : {df["campaign_flag"].sum()}')
assert df['sessions'].min() >= 0
assert df['bounce_rate'].between(0, 1).all()
print('\nAll checks passed.')

## Data Cleaning

The synthetic data requires no cleaning. In a real dataset, we would need to: deduplicate sessions from bot traffic (IP filtering), handle timezone normalization, and cap extreme session counts from traffic spikes.

In [ ]:
df[['sessions', 'pageviews', 'bounce_rate', 'avg_session_duration', 'new_users']].describe().round(2)

## Exploratory Data Analysis

Plot the full sessions time series with rolling average and campaign windows highlighted.

In [ ]:
rolling_7  = df['sessions'].rolling(7, center=True).mean()
rolling_30 = df['sessions'].rolling(30).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df.index, df['sessions'], alpha=0.2, color='steelblue')
ax.plot(df.index, df['sessions'], alpha=0.4, color='steelblue', linewidth=0.7, label='Daily Sessions')
ax.plot(df.index, rolling_7,  color='orange', linewidth=1.5, label='7-Day Rolling Mean')
ax.plot(df.index, rolling_30, color='crimson', linewidth=2, label='30-Day Rolling Mean')

# Shade campaign periods
for start_str, dur in CAMPAIGNS:
    s = pd.Timestamp(start_str)
    e = s + pd.Timedelta(days=dur)
    ax.axvspan(s, e, alpha=0.15, color='green')

green_patch = mpatches.Patch(color='green', alpha=0.3, label='Campaign Period')
ax.legend(handles=ax.lines + [green_patch])
ax.set_title('Daily Website Sessions with Rolling Averages (2022–2023)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
plt.tight_layout()
plt.show()

**Interpretation:** The upward trend in the 30-day rolling mean reflects organic growth. Three distinct spikes (shaded green) correspond to the campaign periods. The 7-day rolling mean captures weekly seasonality smoothly.

## Univariate Analysis

Distribution of daily sessions and identification of outlier days.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['sessions'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df['sessions'].mean(), color='red', linestyle='--', label=f'Mean: {df["sessions"].mean():,.0f}')
axes[0].set_title('Distribution of Daily Sessions')
axes[0].set_xlabel('Sessions')
axes[0].legend()

q1 = df['sessions'].quantile(0.25)
q3 = df['sessions'].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df['sessions'] > q3 + 1.5 * iqr) | (df['sessions'] < q1 - 1.5 * iqr)
axes[1].scatter(df.index, df['sessions'], c=outlier_mask.map({True: 'red', False: 'steelblue'}),
                alpha=0.4, s=8)
axes[1].set_title(f'Sessions with Anomalies Flagged (n={outlier_mask.sum()})')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Sessions')

plt.tight_layout()
plt.show()
print(f'Outlier days: {outlier_mask.sum()} ({outlier_mask.sum()/len(df)*100:.1f}%)')

**Interpretation:** The session distribution is slightly right-skewed due to campaign spike days. IQR-based anomaly detection flags a small percentage of days as outliers — most of these correspond to campaign peaks or viral content events.

## Bivariate / Multivariate Analysis

Relationship between sessions and engagement metrics (bounce rate, session duration).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df['sessions'], df['bounce_rate'], alpha=0.3, s=8, color='steelblue')
m, b, r, p, _ = stats.linregress(df['sessions'], df['bounce_rate'])
x_line = np.linspace(df['sessions'].min(), df['sessions'].max(), 100)
axes[0].plot(x_line, m * x_line + b, color='red', linewidth=2, label=f'r={r:.3f}')
axes[0].set_title('Sessions vs Bounce Rate')
axes[0].set_xlabel('Sessions')
axes[0].set_ylabel('Bounce Rate')
axes[0].legend()

axes[1].scatter(df['sessions'], df['avg_session_duration'], alpha=0.3, s=8, color='tomato')
m2, b2, r2, p2, _ = stats.linregress(df['sessions'], df['avg_session_duration'])
axes[1].plot(x_line, m2 * x_line + b2, color='navy', linewidth=2, label=f'r={r2:.3f}')
axes[1].set_title('Sessions vs Avg Session Duration')
axes[1].set_xlabel('Sessions')
axes[1].set_ylabel('Avg Duration (min)')
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretation:** Higher session days show lower bounce rates and longer session durations, suggesting that traffic surges (especially campaign-driven ones) bring more engaged visitors. This is a positive signal — paid campaigns are attracting quality traffic, not just volume.

## Feature-Specific Insights

Day-of-week and traffic source analysis.

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['dow'] = df.index.dayofweek
dow_avg = df.groupby('dow')['sessions'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(day_names, dow_avg.values, color=sns.color_palette('tab10', 7))
axes[0].set_title('Avg Sessions by Day of Week')
axes[0].set_ylabel('Avg Sessions')
for i, v in enumerate(dow_avg.values):
    axes[0].text(i, v + 50, f'{v:,.0f}', ha='center', fontsize=8)

source_counts = df['source'].value_counts()
axes[1].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('tab10', 4), startangle=140)
axes[1].set_title('Traffic Source Mix')

plt.tight_layout()
plt.show()

**Interpretation:** Monday shows the highest average sessions, consistent with the engineered Monday lift representing business-day content consumption patterns. Weekends have notably lower traffic. Organic search dominates the traffic mix, followed by paid channels — a healthy distribution for long-term growth.

## Statistical Checks

Quantify campaign lift vs. baseline and test whether Monday sessions are significantly higher.

In [ ]:
camp_sessions    = df[df['campaign_flag'] == 1]['sessions']
noncamp_sessions = df[df['campaign_flag'] == 0]['sessions']
camp_lift = (camp_sessions.mean() - noncamp_sessions.mean()) / noncamp_sessions.mean() * 100

t_stat, t_pval = stats.ttest_ind(camp_sessions, noncamp_sessions)
print(f'Campaign sessions avg   : {camp_sessions.mean():,.1f}')
print(f'Non-campaign sessions avg: {noncamp_sessions.mean():,.1f}')
print(f'Campaign lift           : +{camp_lift:.1f}%')
print(f'T-test                  : t={t_stat:.2f}, p={t_pval:.2e}')

# Monday vs rest
mon = df[df['dow'] == 0]['sessions']
rest = df[df['dow'] != 0]['sessions']
t2, p2 = stats.ttest_ind(mon, rest)
print(f'\nMonday avg: {mon.mean():,.1f} | Other days avg: {rest.mean():,.1f}')
print(f'Monday lift: +{(mon.mean()-rest.mean())/rest.mean()*100:.1f}% (p={p2:.4f})')

## Visual Analysis

Monthly sessions heatmap and source trend over time.

In [ ]:
df['month'] = df.index.month
df['year']  = df.index.year
monthly = df.groupby(['year', 'month'])['sessions'].sum().reset_index()
pivot = monthly.pivot(index='year', columns='month', values='sessions')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set_title('Monthly Total Sessions Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation:** The heatmap confirms the upward trend from 2022 to 2023 across all months. Campaign months (March, September, November/December) show notably higher totals in the cells corresponding to the campaign windows.

## Key Findings

In [ ]:
total_sessions = df['sessions'].sum()
top_source = df['source'].value_counts().index[0]
top_source_pct = df['source'].value_counts().iloc[0] / len(df) * 100

print('=== Key Findings ===')
print(f'1. Total sessions (2 years) : {total_sessions:,}')
print(f'2. Avg daily sessions       : {df["sessions"].mean():,.1f}')
print(f'3. Campaign lift            : +{camp_lift:.1f}% vs non-campaign days')
print(f'4. Monday lift              : +{(mon.mean()-rest.mean())/rest.mean()*100:.1f}% vs other days')
print(f'5. Top traffic source       : {top_source} ({top_source_pct:.1f}% of days dominant)')
print(f'6. Avg bounce rate          : {df["bounce_rate"].mean():.3f}')
print(f'7. Avg session duration     : {df["avg_session_duration"].mean():.2f} min')
print(f'8. IQR anomaly days         : {outlier_mask.sum()}')

## Limitations

1. **Source assignment is random** — in reality, source mix varies by day and campaign type.
2. **No conversion tracking** — sessions without conversion data limit business value assessment.
3. **No SEO signals** — real organic traffic depends on keyword rankings and search volume.
4. **Single website** — multi-domain or multi-property analysis is not covered.
5. **Engagement metrics are loosely coupled** — real bounce rate and session duration are influenced by content type, device, and landing page.

## Recommendations / Next Steps

1. Add conversion rate as a metric and analyze campaign ROI (sessions × conversion rate × average order value).
2. Build a traffic forecasting model using Prophet with campaign dummy variables.
3. Segment by device type (mobile/desktop) — mobile often shows higher bounce rates.
4. Add a content calendar overlay to correlate publish dates with traffic spikes.
5. Perform channel attribution modeling to move beyond last-click attribution.

## Common Mistakes

1. Conflating campaign lift with organic growth — always compare to a non-campaign control window.
2. Treating all anomalies as fraud — spikes can be viral content, not just bot traffic.
3. Using page views as a proxy for engagement — session duration and scroll depth are better signals.
4. Averaging bounce rate across sources — each source has a natural baseline rate.
5. Ignoring weekend/weekday differences when computing week-over-week change rates.

## Mini Challenge / Exercises

1. Calculate a 14-day pre/post campaign comparison (week before vs. week after each campaign ends).
2. Plot the bounce rate trend and identify whether it has improved over the two years.
3. Compute the new user rate (new_users / sessions) monthly and check if it is growing or declining.
4. Flag all days where sessions exceed the rolling 30-day mean by more than 2 standard deviations.
5. Build a simple linear regression to predict sessions from day_of_week and campaign_flag.

## Final Summary / Key Takeaways

- Website traffic exhibits clear **day-of-week seasonality** (Monday peak, weekend trough) and a positive **long-term growth trend**.
- **Marketing campaigns** produce measurable session lifts well above the organic baseline, with positive effects on engagement quality metrics.
- **Organic search** dominates the traffic source mix — a strong indicator of SEO health.
- **Anomaly detection** using the IQR method is a fast, practical approach to flag unusual traffic events for investigation.
- Understanding these traffic patterns is essential before building forecasting, attribution, or personalization models.